In [1]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root:", project_root)
print("Python:", sys.executable)

Project root: /Users/ashwinanandram/Documents/GitHub/Multi-Agent Quote Management/rfq_agent
Python: /opt/anaconda3/bin/python


In [2]:
import pandas as pd
import numpy as np

from app.db import engine

In [3]:
df = pd.read_sql(
    "SELECT * FROM bid_training_data",
    engine
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 10000
Columns: 27


In [64]:
print(df.columns.tolist())

['bid_id', 'customer_id', 'industry', 'country', 'segment', 'sales_rep_id', 'experience_years', 'bid_date', 'total_cost', 'quote_price', 'discount_pct', 'margin_pct', 'deal_size', 'competitor_present', 'competitor_price', 'days_to_quote', 'product_id', 'product_category', 'product_unit_cost', 'product_list_price', 'customer_prior_bids', 'customer_prior_wins', 'customer_prior_losses', 'customer_historical_win_rate', 'customer_avg_prior_deal_size', 'customer_avg_prior_discount', 'won']


In [65]:
df["quote_to_cost_ratio"] = (
    df["quote_price"] / df["total_cost"]
)

df["discount_amount"] = (
    df["deal_size"] - df["quote_price"]
)

df["deal_size_log"] = (
    np.log1p(df["deal_size"])
)

In [66]:
print(
    df[
        [
            "quote_to_cost_ratio",
            "discount_amount",
            "deal_size_log"
        ]
    ].head()
)

   quote_to_cost_ratio  discount_amount  deal_size_log
0                1.840          9393.24      11.673482
1                1.488         50430.64      13.487616
2                2.185          1975.08      10.584121
3                1.204         37164.87      12.489236
4                1.320         35855.49      12.607519


In [67]:
numeric_features_v4 = [
    "total_cost",
    "quote_price",
    "discount_pct",
    "margin_pct",
    "deal_size",
    "experience_years",
    "product_unit_cost",
    "product_list_price",
    "quote_to_cost_ratio",
    "discount_amount",
    "deal_size_log",
    "customer_prior_bids",
    "customer_prior_wins",
    "customer_prior_losses",
    "customer_historical_win_rate",
    "customer_avg_prior_deal_size",
    "customer_avg_prior_discount"
]

categorical_features_v4 = [
    "industry",
    "country",
    "segment",
    "product_category",
    "competitor_present"
]

required_features = (
    numeric_features_v4 +
    categorical_features_v4
)

missing_features = [
    column
    for column in required_features
    if column not in df.columns
]

print("Number of V4 features:", len(required_features))
print("Missing features:", missing_features)

Number of V4 features: 22
Missing features: []


In [68]:
X = df[required_features]
y = df["won"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 22)
y shape: (10000,)


In [69]:
df["bid_date"] = pd.to_datetime(df["bid_date"])

print("Earliest bid:", df["bid_date"].min())
print("Latest bid:", df["bid_date"].max())

Earliest bid: 2024-01-01 00:00:00
Latest bid: 2026-10-06 00:00:00


In [70]:
cutoff_date = pd.Timestamp("2026-03-15")

train_mask = df["bid_date"] < cutoff_date
test_mask = df["bid_date"] >= cutoff_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 7994
Test rows: 2006


In [71]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder


numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)


preprocessor_v4 = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features_v4
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features_v4
        )
    ]
)

In [72]:
from xgboost import XGBClassifier


xgb_model_v4 = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_v4
        ),
        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=5,
                random_state=42,
                eval_metric="logloss",
                n_jobs=-1
            )
        )
    ]
)

In [73]:
xgb_model_v4.fit(
    X_train,
    y_train
)

print("V4 model training completed.")

V4 model training completed.


In [74]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)


y_pred = xgb_model_v4.predict(X_test)

y_prob = xgb_model_v4.predict_proba(
    X_test
)[:, 1]


accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)


print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("ROC-AUC:", roc_auc)

Accuracy: 0.5797607178464607
Precision: 0.5301047120418848
Recall: 0.45556805399325084
ROC-AUC: 0.5831847115798081


In [75]:
from sklearn.metrics import confusion_matrix


cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[758 359]
 [484 405]]


In [76]:
from pathlib import Path
import joblib


model_dir = Path("../ml/models")

model_dir.mkdir(
    parents=True,
    exist_ok=True
)

model_path = model_dir / "xgb_v4.joblib"


joblib.dump(
    xgb_model_v4,
    model_path
)


print("Model saved to:")
print(model_path.resolve())

Model saved to:
/Users/ashwinanandram/Documents/GitHub/Multi-Agent Quote Management/rfq_agent/ml/models/xgb_v4.joblib


In [6]:
from app.agents.research_agent import ResearchAgent
from app.azure_client import create_azure_openai_client
from app.config import get_required_setting

client = create_azure_openai_client()

deployment_name = get_required_setting(
    "AZURE_OPENAI_DEPLOYMENT"
)

research_agent = ResearchAgent(
    client=client,
    deployment_name=deployment_name
)

research_agent.run(
    bid_id=10001,
    user_message="What is this customer's historical win rate?"
)

"The customer's historical win rate is 50% (0.50). This means that historically, the customer has won 18 out of 36 prior bids. If you need additional historical details about this customer, please let me know."

In [7]:
from app.agents.pricing_agent import PricingAgent
from app.azure_client import create_azure_openai_client
from app.config import get_required_setting

client = create_azure_openai_client()

deployment_name = get_required_setting(
    "AZURE_OPENAI_DEPLOYMENT"
)

pricing_agent = PricingAgent(
    client=client,
    deployment_name=deployment_name
)

pricing_agent.run(
    bid_id=10001,
    user_message="What is the optimal price?"
)

'The optimal price for bid ID 10001 is $114,207.72. \n\nAt this price, the discount is approximately 2.73%, which is well within the maximum allowed discount of 25%. The margin achieved is about 48.6%, exceeding the minimum margin requirement of 35%. \n\nThis pricing results in a win probability of approximately 21.5%, with an expected profit of $11,911.02. \n\nTherefore, setting the quote price to $114,207.72 optimizes the expected profit while complying with the pricing policy constraints.'

In [14]:
from app.agents.orchestration_agent import build_orchestration_graph

graph = build_orchestration_graph()

result = graph.invoke({
    "bid_id": 10001,
    "user_message": "Compare prices of $100150, $110000 & $95000?"
})

result

{'bid_id': 10001,
 'user_message': 'Compare prices of $100150, $110000 & $95000?',
 'router_decision': 'PROCEED',
 'selected_agents': ['pricing'],
 'execution_plan': ['pricing'],
 'completed_agents': ['pricing'],
 'agent_results': {'pricing': {'status': 'COMPLETED',
   'response': 'Here is the comparison of the three proposed quote prices for Bid ID 10001:\n\n- At $100,150:\n  - Discount: 14.7%\n  - Margin: 41.38%\n  - Win Probability: 23.3%\n  - Expected Profit: $9,663\n\n- At $110,000:\n  - Discount: 6.3%\n  - Margin: 46.63%\n  - Win Probability: 20.3%\n  - Expected Profit: $10,398\n\n- At $95,000:\n  - Discount: 19.1%\n  - Margin: 38.2%\n  - Win Probability: 24.1%\n  - Expected Profit: $8,758\n\nSummary:\n- The highest expected profit is at the $110,000 price, but it has the lowest win probability.\n- The highest win probability is at the $95,000 price, but it yields the lowest expected profit.\n- The $100,150 price offers a balance with moderate win probability and expected profit.

In [3]:
from app.agents.tools import risk_evidence_service

risk = risk_evidence_service.evaluate(
    bid_id=10001,
    proposed_quote_price=114207.72
)

risk

{'bid_id': 10001,
 'quote': {'current_quote_price': 108022.2,
  'proposed_quote_price': 114207.72,
  'deal_size': 117415.44,
  'margin_pct': 48.59566411097253,
  'discount_pct': 2.7319405352481763,
  'win_probability': 0.21461305022239685},
 'deal_size_band': {'deal_size_risk_band_id': 2,
  'band_name': '100K_TO_500K',
  'min_deal_size': 100000.0,
  'max_deal_size': 500000.0},
 'margin': {'margin_pct': 48.59566411097253,
  'minimum_margin_pct': 35.0,
  'risk': 'MEDIUM'},
 'discount': {'discount_pct': 2.7319405352481763, 'risk': 'MEDIUM'},
 'win_probability': {'probability': 0.21461305022239685, 'risk': 'MEDIUM'},
 'overall_risk': 'MEDIUM'}

In [3]:
from app.azure_client import create_azure_openai_client
from app.config import get_required_setting
from app.agents.risk_agent import RiskAgent

client = create_azure_openai_client()

risk_agent = RiskAgent(
    client=client,
    deployment_name=get_required_setting("AZURE_OPENAI_DEPLOYMENT"),
)

In [4]:
risk_result = risk_agent.run(
    bid_id=10001,
    proposed_quote_price=114207.72,
)

risk_result

{'bid_id': 10001,
 'overall_risk': 'MEDIUM',
 'assessment': 'The overall risk for the bid is classified as MEDIUM. The margin percentage is 48.60%, which falls under the MEDIUM risk classification. The discount percentage is 2.73%, also classified as MEDIUM risk. The win probability is 21.46%, and this value is likewise categorized as MEDIUM risk. The deal size is within the 100K_TO_500K band, identified as deal size risk band 2. The primary concerns are margin risk, discount risk, and win probability risk, each identified as MEDIUM risk according to the supplied evidence.',
 'primary_concerns': ['Margin risk at 48.60% classified as MEDIUM',
  'Discount risk at 2.73% classified as MEDIUM',
  'Win probability risk at 21.46% classified as MEDIUM'],
 'evidence': {'bid_id': 10001,
  'quote': {'current_quote_price': 108022.2,
   'proposed_quote_price': 114207.72,
   'deal_size': 117415.44,
   'margin_pct': 48.59566411097253,
   'discount_pct': 2.7319405352481763,
   'win_probability': 0.214

In [4]:
import importlib
import app.agents.orchestration_agent

importlib.reload(app.agents.orchestration_agent)

<module 'app.agents.orchestration_agent' from '/Users/ashwinanandram/Documents/GitHub/Multi-Agent Quote Management/rfq_agent/app/agents/orchestration_agent.py'>

In [5]:
from app.agents.orchestration_agent import build_orchestration_graph

graph = build_orchestration_graph()

In [7]:
result = graph.invoke({
    "bid_id": 10001,
    "user_message": "What is the risk associated with bid at a price of $112000?"
})

result

{'bid_id': 10001,
 'user_message': 'What is the risk associated with bid at a price of $112000?',
 'router_decision': 'PROCEED',
 'selected_agents': ['risk', 'pricing'],
 'execution_plan': ['risk', 'pricing'],
 'completed_agents': ['pricing', 'risk'],
 'agent_results': {'pricing': {'status': 'COMPLETED',
   'response': 'For the proposed bid price of $112,000 on Bid ID 10001, the key pricing metrics indicate a discount of approximately 4.61% and a margin of about 47.58%. \n\nThe associated risk can be assessed through the win probability, which is 18.0%. This relatively low win probability suggests a higher chance of losing the bid at this price point.\n\nHowever, the expected profit from this bid is $9,595.69, reflecting the profit you can anticipate if the bid is won.\n\nIn summary, the risk is primarily in the lower likelihood of winning the bid (18%) despite the higher price. This means while the margin and profit are attractive, the chance of losing the bid is significant at this p

In [5]:

import json

from app.agents.orchestration_agent import client, deployment_name
from app.services.quote_scenario_resolver import resolve_quote_scenario

test_cases = [
    {
        "bid_id": 10001,
        "user_message": "What is the risk associated with bid at 10% discount on the current quote?",
    },
    {
        "bid_id": 10001,
        "user_message": "Compare quotes of $110K and $115K at 5% discount.",
    },
    {
        "bid_id": 10001,
        "user_message": "Give me two best optimal prices that maximize expected profit?",
    },
]

for test_case in test_cases:
    result = resolve_quote_scenario(
        **test_case,
        client=client,
        deployment_name=deployment_name,
    )
    print(json.dumps(result, indent=2))


/opt/anaconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{
  "intent": "DISCOUNT_SCENARIO",
  "proposed_quote_prices": [
    105480.0
  ],
  "discount_pct": 10.0,
  "discount_basis": "CURRENT_QUOTE",
  "optimization_count": 1,
  "requested_analysis": [
    "risk"
  ],
  "clarification_question": null,
  "reasoning": "The user is asking about the risk involved with applying a 10% discount to the current quote. This fits the DISCOUNT_SCENARIO intent with discount_pct of 10 and discount_basis on the current quote, with a focus on risk analysis.",
  "current_quote_price": 117200.0
}
{
  "intent": "DISCOUNT_SCENARIO",
  "proposed_quote_prices": [
    104500.0,
    109250.0
  ],
  "discount_pct": 5.0,
  "discount_basis": "EACH_PROPOSED_PRICE",
  "optimization_count": 1,
  "requested_analysis": [
    "pricing",
    "recommendation"
  ],
  "clarification_question": null,
  "reasoning": "User wants to compare two specific proposed quote prices with a 5% discount applied to each price to analyze pricing and obtain recommendations."
}
{
  "intent": "OP

In [4]:
from app.agents.tools import simulate_quote

result = simulate_quote(
    bid_id=10001,
    proposed_quote_price=105480.0
)

print(result)

{'status': 'VALID', 'result': {'bid_id': 10001, 'current_quote_price': 117200.0, 'proposed_quote_price': 105480.0, 'discount_pct': 10.0, 'margin_pct': 44.34232081911263, 'win_probability': 0.23020650446414948, 'expected_profit': 10767.28308461845, 'status': 'VALID', 'validation_error': None}}


In [3]:

from app.agents.orchestration_agent import orchestrate, run_pricing

test_cases = [
    {
        "bid_id": 10001,
        "user_message": "Compare quotes of $95K, $110K and $115K at 5% discount.",
    },
    {
        "bid_id": 10001,
        "user_message": "What is the risk associated with bid at 10% discount on the current quote?",
    },
    {
        "bid_id": 10001,
        "user_message": "Give me two best optimal prices that maximize expected profit?",
    },
]

for test_case in test_cases:
    state = orchestrate(test_case)
    scenario = state.get("quote_scenario", {})

    print("\nRequest:", test_case["user_message"])
    print("Resolved intent:", scenario.get("intent"))
    print("Resolved prices:", scenario.get("proposed_quote_prices"))
    print("Execution plan:", state.get("execution_plan"))

    if "pricing" in state.get("execution_plan", []):
        pricing_state = run_pricing(state)
        print("Pricing status:", pricing_state["agent_results"]["pricing"]["status"])
        print("Pricing response:", pricing_state["response"])


/opt/anaconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



Request: Compare quotes of $95K, $110K and $115K at 5% discount.
Resolved intent: PRICE_COMPARISON
Resolved prices: [95000.0, 110000.0, 115000.0]
Execution plan: ['pricing']
Pricing status: COMPLETED
Pricing response: Here is the comparison of the three proposed quote prices with a 5% discount:

1. Quote at $95,000:
   - Discount: ~18.94% (higher than the requested 5%)
   - Margin: 38.20%
   - Win Probability: 23.6%
   - Expected Profit: $8,569

2. Quote at $110,000:
   - Discount: ~6.14% (close to the requested 5%)
   - Margin: 46.63%
   - Win Probability: 20.3%
   - Expected Profit: $10,398

3. Quote at $115,000:
   - Discount: ~1.88% (less than the requested 5%)
   - Margin: 48.95%
   - Win Probability: 19.2%
   - Expected Profit: $10,835

Summary:
- The $95K quote has the highest win probability (23.6%) but the lowest expected profit ($8,569).
- The $115K quote has the lowest win probability (19.2%) but the highest expected profit ($10,835).
- The $110K quote strikes a balance wit

In [4]:
import inspect
from app.services.quote_scenario_resolver import QuoteScenarioResolver

print(inspect.getsource(QuoteScenarioResolver.resolve))
print(inspect.getsource(QuoteScenarioResolver._normalize_current_quote_discount))

    def resolve(
        self,
        bid_id: int,
        user_message: str,
    ) -> dict:
        response = self.client.responses.parse(
            model=self.deployment_name,
            instructions=(
                "You are a quote scenario resolver for an enterprise "
                "RFQ and quote management system. Classify the user's "
                "request into a QuoteScenario. Do not invent bid data "
                "or calculate prices. "
                "Use SPECIFIC_PRICE for one explicit proposed price. "
                "Use PRICE_COMPARISON for two or more explicit prices "
                "to compare. Use DISCOUNT_SCENARIO when a discount is "
                "requested. Set discount_basis to CURRENT_QUOTE when "
                "the discount applies to the current or existing quote, "
                "or EACH_PROPOSED_PRICE when it applies to explicitly "
                "provided prices. Use OPTIMIZATION when the user asks "
                "for the best or

In [3]:
from app.agents.orchestration_agent import client, deployment_name
from app.services.quote_scenario_resolver import resolve_quote_scenario

result = resolve_quote_scenario(
    bid_id=10001,
    user_message="Compare quotes of $95K, $110K and $115K at 5% discount.",
    client=client,
    deployment_name=deployment_name,
)

print(result["intent"])
print(result["proposed_quote_prices"])
print(result["discount_pct"])
print(result["discount_basis"])

/opt/anaconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


DISCOUNT_SCENARIO
[90250.0, 104500.0, 109250.0]
5.0
EACH_PROPOSED_PRICE


In [4]:
from app.agents.orchestration_agent import orchestrate, run_pricing

test_case = {
    "bid_id": 10001,
    "user_message": "Compare quotes of $95K, $110K and $115K at 5% discount.",
}

state = orchestrate(test_case)

print("Intent:", state["quote_scenario"]["intent"])
print("Adjusted prices:", state["quote_scenario"]["proposed_quote_prices"])
print("Execution plan:", state["execution_plan"])

pricing_state = run_pricing(state)

print("Pricing status:", pricing_state["agent_results"]["pricing"]["status"])
print("Pricing response:", pricing_state["response"])

Intent: DISCOUNT_SCENARIO
Adjusted prices: [90250.0, 104500.0, 109250.0]
Execution plan: ['pricing']
Pricing status: COMPLETED
Pricing response: For Bid ID 10001, comparing the proposed quotes at 5% discount applied to your requested prices ($95K, $110K, and $115K):

- The $95K quote (effectively $90,250 after discount) yields a discount of ~23%, margin of ~35%, win probability 35%, and expected profit of about $11,054.
- The $110K quote (effectively $104,500 after discount) yields a discount of ~11%, margin of ~44%, win probability 25%, and expected profit of about $11,317.
- The $115K quote (effectively $109,250 after discount) yields a discount of ~7%, margin of ~46%, win probability 20%, and expected profit of about $10,246.

Summary:
- The lowest proposed quote maximizes win probability (35%) and keeps expected profit above $11K.
- The mid quote trades off some win probability for slightly higher expected profit (~$11,317).
- The highest quote has the best margin but lowest win pr

In [5]:
from app.agents.tools import compare_prices

result = compare_prices(
    bid_id=10001,
    proposed_quote_prices=[90250.0, 104500.0, 109250.0],
)

print(result)

{'status': 'COMPLETED', 'bid_id': 10001, 'comparisons': [{'proposed_quote_price': 90250.0, 'status': 'VALID', 'result': {'bid_id': 10001, 'current_quote_price': 117200.0, 'proposed_quote_price': 90250.0, 'discount_pct': 22.994880546075088, 'margin_pct': 34.949894736842104, 'win_probability': 0.35045304894447327, 'expected_profit': 11054.08819666028, 'status': 'VALID', 'validation_error': None}}, {'proposed_quote_price': 104500.0, 'status': 'VALID', 'result': {'bid_id': 10001, 'current_quote_price': 117200.0, 'proposed_quote_price': 104500.0, 'discount_pct': 10.836177474402731, 'margin_pct': 43.82036363636364, 'win_probability': 0.24714332818984985, 'expected_profit': 11317.256484601498, 'status': 'VALID', 'validation_error': None}}, {'proposed_quote_price': 109250.0, 'status': 'VALID', 'result': {'bid_id': 10001, 'current_quote_price': 117200.0, 'proposed_quote_price': 109250.0, 'discount_pct': 6.783276450511945, 'margin_pct': 46.26295652173913, 'win_probability': 0.20272374153137207, 

In [3]:
import inspect
from app.agents.tools import optimization_service

print(inspect.signature(type(optimization_service).optimize))

(self, bid: dict, pricing_policy: dict, min_price: float | None = None, max_price: float | None = None, step: float = 1000.0, top_n: int = 1) -> dict


In [6]:
from app.agents.orchestration_agent import orchestrate, run_pricing

test_case = {
    "bid_id": 10001,
    "user_message": "Give me five best optimal prices that maximize expected profit?"
}

state = orchestrate(test_case)
pricing_state = run_pricing(state)

print(pricing_state["agent_results"]["pricing"])

{'status': 'COMPLETED', 'response': 'The five best optimal prices for Bid ID 10001 that maximize expected profit are:\n\n1. Proposed Price: 114,207.72 with 2.55% discount, margin 48.60%, win probability 21.46%, expected profit $11,911.02.\n2. Proposed Price: 113,707.72 with 2.98% discount, margin 48.37%, win probability 21.46%, expected profit $11,803.72.\n3. Proposed Price: 113,207.72 with 3.41% discount, margin 48.14%, win probability 21.46%, expected profit $11,696.41.\n4. Proposed Price: 112,707.72 with 3.83% discount, margin 47.91%, win probability 21.46%, expected profit $11,589.10.\n5. Proposed Price: 117,415.44 with -0.18% discount (a slight price increase), margin 50.00%, win probability 19.67%, expected profit $11,545.94.\n\nAll these options comply with the pricing policy constraints of a minimum margin of 35% and maximum discount of 25%. The top option achieves the highest expected profit by balancing a good margin with a reasonable win probability.'}


In [8]:
import math
import pandas as pd

from app.agents.tools import get_bid_simulation_context_data
from app.services.ml_prediction import MLPredictionService

bid = get_bid_simulation_context_data(10001)
service = MLPredictionService()

prices = [
    114207.72,
    113707.72,
    113207.72,
    112707.72,
    117415.44,
]

for price in prices:
    total_cost = float(bid["total_cost"])
    deal_size = float(bid["deal_size"])
    current_quote = float(bid["quote_price"])

    features = {
        "total_cost": total_cost,
        "quote_price": price,
        "discount_pct": (current_quote - price) / current_quote * 100,
        "margin_pct": (price - total_cost) / price * 100,
        "deal_size": deal_size,
        "experience_years": bid["experience_years"],
        "product_unit_cost": bid["product_unit_cost"],
        "product_list_price": bid["product_list_price"],
        "quote_to_cost_ratio": price / total_cost,
        "discount_amount": current_quote - price,
        "deal_size_log": math.log1p(deal_size),
        "customer_prior_bids": bid["customer_prior_bids"],
        "customer_prior_wins": bid["customer_prior_wins"],
        "customer_prior_losses": bid["customer_prior_losses"],
        "customer_historical_win_rate": bid["customer_historical_win_rate"],
        "customer_avg_prior_deal_size": bid["customer_avg_prior_deal_size"],
        "customer_avg_prior_discount": bid["customer_avg_prior_discount"],
        "industry": bid["industry"],
        "country": bid["country"],
        "segment": bid["segment"],
        "product_category": bid["product_category"],
        "competitor_present": bid["competitor_present"],
    }

    probability = service.predict_win_probability(features)

    print(
        f"Price: ${price:,.2f} | "
        f"Discount: {features['discount_pct']:.4f}% | "
        f"Margin: {features['margin_pct']:.4f}% | "
        f"Win probability: {probability:.8f}"
    )

Price: $114,207.72 | Discount: 2.5531% | Margin: 48.5957% | Win probability: 0.21461305
Price: $113,707.72 | Discount: 2.9798% | Margin: 48.3696% | Win probability: 0.21461305
Price: $113,207.72 | Discount: 3.4064% | Margin: 48.1416% | Win probability: 0.21461305
Price: $112,707.72 | Discount: 3.8330% | Margin: 47.9115% | Win probability: 0.21461305
Price: $117,415.44 | Discount: -0.1838% | Margin: 50.0000% | Win probability: 0.19666813


In [3]:
from app.agents.orchestration_agent import build_orchestration_graph

graph = build_orchestration_graph()

test_messages = [
    "Simulate a quote at $110,000 and assess the risk.",
    "Compare quotes of $95,000, $110,000 and $115,000.",
    "Give me the two best prices that maximize expected profit.",
    "What are the main risks associated with this bid?"
]

for message in test_messages:
    result = graph.invoke({
        "bid_id": 10001,
        "user_message": message
    })

    print("\n" + "=" * 80)
    print("REQUEST:", message)
    print("INTENT:", result.get("quote_scenario", {}).get("intent"))
    print("EXECUTION PLAN:", result.get("execution_plan"))
    print("COMPLETED AGENTS:", result.get("completed_agents"))
    print("FINAL RESPONSE:", result.get("response"))

/opt/anaconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



REQUEST: Simulate a quote at $110,000 and assess the risk.
INTENT: SPECIFIC_PRICE
EXECUTION PLAN: ['pricing', 'risk']
COMPLETED AGENTS: ['pricing', 'risk']
FINAL RESPONSE: For the bid ID 10001, the proposed quote price of $110,000 was assessed.

Risk classification at $110,000:
- Overall risk: HIGH
- Discount risk: HIGH (6.14% discount)
- Margin risk: MEDIUM (46.63% margin)
- Win probability risk: MEDIUM (20.27% win probability)

This means the $110,000 quote carries a high overall risk primarily due to the discount level, with medium risks associated with margin and win probability.

REQUEST: Compare quotes of $95,000, $110,000 and $115,000.
INTENT: PRICE_COMPARISON
EXECUTION PLAN: ['pricing']
COMPLETED AGENTS: ['pricing']
FINAL RESPONSE: Here is the comparison of the three proposed quote prices for Bid ID 10001:

1. At $95,000:
- Discount: 18.94%
- Margin: 38.2%
- Win Probability: 23.6%
- Expected Profit: $8,569.22

2. At $110,000:
- Discount: 6.14%
- Margin: 46.63%
- Win Probabilit

In [14]:
import inspect
import app.agents.orchestration_agent as orchestration

print("=== risk_agent.run ===")
print(inspect.getsource(orchestration.risk_agent.run))

print("\n=== Risk evidence evaluation ===")
from app.services.risk_evidence import RiskEvidenceService
print(inspect.getsource(RiskEvidenceService.evaluate))

=== risk_agent.run ===
    def run(
        self,
        bid_id: int,
        proposed_quote_price: float | None = None,
    ) -> dict:

        evidence = risk_evidence_service.evaluate(
            bid_id=bid_id,
            proposed_quote_price=proposed_quote_price,
        )

        response = self.client.responses.parse(
            model=self.deployment_name,
            instructions=(
                "You are a sales deal risk analyst. "
                "Your role is to interpret authoritative risk evidence provided by the "
                "RiskEvidenceService and explain the risk profile of the proposed quote. "

                "The RiskEvidenceService is the sole authority for all risk classifications. "
                "Use the supplied margin risk, discount risk, win probability risk, "
                "deal-size risk band, and overall risk exactly as provided. "
                "Never recalculate, reinterpret, override, or change these classifications. "

              

In [15]:
from app.agents.tools import get_bid_simulation_context_data

bid = get_bid_simulation_context_data(10001)

print("quote_price:", bid.get("quote_price"))
print("deal_size:", bid.get("deal_size"))
print("total_cost:", bid.get("total_cost"))

quote_price: 117200.00
deal_size: 117415.44
total_cost: 58707.72


In [5]:
import app.agents.risk_agent as risk_module

print("Service object:", risk_module.risk_evidence_service)
print("Service type:", type(risk_module.risk_evidence_service))

evidence = risk_module.risk_evidence_service.evaluate(
    bid_id=10001,
    proposed_quote_price=None,
)

print("\nRisk evidence:")
print(evidence)

Service object: <app.services.risk_evidence.RiskEvidenceService object at 0x154f7a120>
Service type: <class 'app.services.risk_evidence.RiskEvidenceService'>

Risk evidence:
{'bid_id': 10001, 'quote': {'current_quote_price': 117200.0, 'proposed_quote_price': 117200.0, 'deal_size': 117415.44, 'margin_pct': 49.90808873720136, 'discount_pct': 0.0, 'win_probability': 0.1966681331396103}, 'deal_size_band': {'deal_size_risk_band_id': 2, 'band_name': '100K_TO_500K', 'min_deal_size': 100000.0, 'max_deal_size': 500000.0}, 'margin': {'margin_pct': 49.90808873720136, 'minimum_margin_pct': 35.0, 'risk': 'MEDIUM'}, 'discount': {'discount_pct': 0.0, 'risk': 'LOW'}, 'win_probability': {'probability': 0.1966681331396103, 'risk': 'HIGH'}, 'overall_risk': 'MEDIUM'}


In [3]:
from app.agents.orchestration_agent import build_orchestration_graph

graph = build_orchestration_graph()

result = graph.invoke({
    "bid_id": 10001,
    "user_message": "Compare quotes of $95,000, $110,000 and $115,000 at a 5% discount, and assess the risk."
})

print("Intent:", result.get("quote_scenario", {}).get("intent"))
print("Adjusted prices:", result.get("quote_scenario", {}).get("proposed_quote_prices"))
print("Execution plan:", result.get("execution_plan"))
print("Completed agents:", result.get("completed_agents"))
print("\nFinal response:")
print(result.get("response"))

/opt/anaconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Intent: DISCOUNT_SCENARIO
Adjusted prices: [90250.0, 104500.0, 109250.0]
Execution plan: ['pricing', 'risk']
Completed agents: ['pricing', 'risk']

Final response:
For Bid ID 10001, the risk assessments correspond to the proposed prices after applying a 5% discount:

1. Adjusted price: $90,250 (from $95,000)  
   - Risk classification: HIGH overall risk  
   - Reasons: HIGH margin risk (margin 34.95% below 35% minimum), HIGH discount risk (22.99%), MEDIUM win probability risk (35.05%)  

2. Adjusted price: $104,500 (from $110,000)  
   - Risk classification: HIGH overall risk  
   - Reasons: HIGH discount risk (10.84%), MEDIUM margin risk (43.82%), MEDIUM win probability risk (24.7%)  

3. Adjusted price: $109,250 (from $115,000)  
   - Risk classification: HIGH overall risk  
   - Reasons: HIGH discount risk (6.78%), MEDIUM margin risk (46.26%), MEDIUM win probability risk (20.27%)  

All three proposed prices with a 5% discount were assessed for risk. Each carries an overall HIGH ris